# Train Relevia's own model (for the app)

Run top to bottom on a **Colab GPU** (Runtime → Change runtime type → T4 or L4).

**Important:** the app already works well with the plain Llama-3.2-3B. Only ship the fine-tune if step 5 shows it's clearly better.

| Step | Time (3B, T4) |
|---|---|
| 1–2 setup + data | ~5 min |
| 3 learning-rate sweep | ~1–1.5 h |
| 4 full training | ~2–3 h |
| 5 side-by-side check | ~15 min |
| 6 export to the app | ~20 min |

## 1. Setup
Upload `relevia-research` to Google Drive first, or clone it from GitHub. Accept Meta's licence on https://huggingface.co/meta-llama/Llama-3.2-3B-Instruct before logging in.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
%cd /content/drive/MyDrive/relevia-research
!pip install -q -r requirements.txt

In [ ]:
from huggingface_hub import notebook_login
notebook_login()   # use a token with WRITE access if you want step 6 to upload

## 2. Data

In [ ]:
!python data/build_dataset.py

In [ ]:
MODEL = 'configs/models/llama-3.2-3b.yaml'

## 3. Find the learning rate
Short runs at 5e-5, 1e-4, 2e-4 and 3e-4. At the end it prints which one to use. **Put that number in `configs/base.yaml` → `train.lr`** before step 4 (it's 2e-4 right now).

In [ ]:
!python src/lr_sweep.py --model {MODEL}

## 4. Full training

In [ ]:
!python src/train.py --model {MODEL} --seed 1

## 5. Is it actually better?
Compare it with the plain model using the app-style prompt on 40 val messages. Open `results/side_by_side_tuned_prompt_vs_finetuned_tuned.md`, choose the better reply for each pair, *then* open the key.

**Rule:** ship only if the fine-tune wins clearly (for example 25+ of 40). If not, keep the app on the plain model, which is still a good result.

In [ ]:
!python src/generate.py --model {MODEL} --condition tuned_prompt --testset val --limit 40
!python src/generate.py --model {MODEL} --condition finetuned_tuned --testset val --limit 40
!python src/side_by_side.py --model llama-3.2-3b --a tuned_prompt --b finetuned_tuned --testset val

In [ ]:
# also check it didn't get worse at safety
!python src/generate.py --model {MODEL} --condition finetuned_tuned --testset safety/crisis
!python src/generate.py --model {MODEL} --condition tuned_prompt --testset safety/crisis
!python src/evaluate.py
import pandas as pd; pd.read_csv('results/metrics/auto_metrics.csv')[['condition','testset','crisis_referral_rate','false_alarm_rate']].dropna()

## 6. Export to the app (only if step 5 says ship)
Replace `<you>` with your Hugging Face username. At the end it prints a `CUSTOM_MODEL` block. Paste it into `relevia/src/services/modelConfig.js`.

In [ ]:
!pip install -q --pre -U -f https://mlc.ai/wheels mlc-llm-nightly-cu123 mlc-ai-nightly-cu123
!python deploy/export_to_webllm.py --model {MODEL} --seed 1 --hf-repo <you>/Relevia-3B-ft-q4f16_1-MLC